In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import numpy as np

from Lab3.MNIST_MLP import criterion

mnist = fetch_openml('mnist_784', version=1, as_frame=False) # X: (70000, 784)
X, y = mnist["data"], mnist["target"]
X = X.astype(np.float32) / 255.0
y = y.astype(np.int64)
# scale to [0, 1]
X_trainval, X_test, y_trainval, y_test = train_test_split(
X, y, test_size=10000, random_state=42, stratify=y
)


# MLP Model

In [ ]:
import os
import torch
from torch import nn

class MLP_Model(nn.Module):
    def __init__(self):
        super(MLP_Model, self).__init__()
        self.input = nn.Linear(784,2 * 784 )
        self.hidden = nn.Linear(2*784, 2*784)
        self.output = nn.Linear(2*784, 10)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.input(x))
        x = self.relu(self.hidden(x))
        x = self.output(x)

        return x


os.makedirs('Models', exist_ok=True)

# optuna for Learning Rate

In [ ]:
import optuna
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, TensorDataset
from sklearn.model_selection import KFold

# --- Data ---
X_trainval_tensor = torch.from_numpy(X_trainval).float()
y_trainval_tensor = torch.from_numpy(y_trainval).long()
dataset = TensorDataset(X_trainval_tensor, y_trainval_tensor)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_SPLITS = 5
N_EPOCHS = 5
BATCH    = 1024


def _train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        criterion(model(X), y).backward()
        optimizer.step()

def _calculate_val_loss(model, loader, criterion):
    model.eval()

    running_loss = 0.0
    with torch.no_grad():
        for X,y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            output = model(X)
            loss = criterion(output, y)
            running_loss += loss.item()
    avg_loss = running_loss / len(loader)

    return avg_loss

def objetive_lr(trial:optuna.Trial) -> float:
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-1, log=True)

    criterion = nn.CrossEntropyLoss()
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    fold_loss = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_trainval_tensor)):
        train_loader = DataLoader(Subset(dataset, train_idx), batch_size=BATCH, shuffle=True)
        val_loader = DataLoader(Subset(dataset, val_idx), batch_size=len(val_idx), shuffle=False)

        model = MLP_Model().to(DEVICE)
        optimizer = optim.SGD(model.parameters(), lr=lr)

        for epoch in range(N_EPOCHS):
            _train_one_epoch(model=model, loader=train_loader, optimizer=optimizer, criterion=criterion)

        fold_loss.append(_calculate_val_loss(model=model, loader=val_loader, criterion=criterion))